# Module 6: Model Packaging, REST API & Cloud Deployment

## Overview
This notebook demonstrates the end-to-end packaging and deployment workflow for the **Ames Housing Price Prediction** model (`LGBMRegressor`, 60 selected features) using modular utilities from `src/deployment.py`.

### Key Objectives
1. **Model Serialization & Calibration:** Fit the production regressor on `X_train_selected.csv`, compute out-of-fold log-residual statistics ($\sigma_{\log}$ and empirical 95% quantiles) for calibrated confidence intervals, and save `models/ames_housing_bundle.joblib` plus `models/model_metadata.json`.
2. **Local Inference Verification:** Test `predict_with_confidence()` across representative property profiles (median home, luxury remodel, compact starter home) and inspect point estimates alongside prediction intervals.
3. **Deployment Artifact Generation:** Verify and inspect the generated FastAPI service (`app/api.py`), interactive Streamlit UI (`app/streamlit_app.py`), container configuration (`app/Dockerfile`), and Render Blueprint (`app/render.yaml`).
4. **API Smoke Testing:** Run automated endpoint checks against the FastAPI application (`/health`, `/features`, `/predict`) using FastAPI's `TestClient` (as well as optional live cloud endpoint checks).

## 1. Environment Setup & Project Imports

In [1]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd

# Resolve repository root cleanly regardless of whether the notebook is run from root or notebooks/
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
for p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, 'src')):
    if os.path.exists(p) and p not in sys.path:
        sys.path.insert(0, p)

from src.deployment import (
    build_and_save_model_bundle,
    predict_with_confidence,
    write_deployment_files,
    run_packaging_pipeline,
)

print(f'Repository root resolved: {os.path.basename(PROJECT_ROOT)}')

Repository root resolved: real-estate-price-prediction


## 2. Model Packaging & Confidence Calibration (`src/deployment.py`)

We load the leak-free selected training matrix (`data/processed/X_train_selected.csv`) and log-transformed target (`data/processed/y_train_log.csv`), run 5-fold cross-validation to calibrate the residual standard deviation $\sigma_{\log}$, and serialize the complete inference bundle to `models/ames_housing_bundle.joblib`.

In [2]:
data_dir = os.path.join(PROJECT_ROOT, 'data', 'processed')
models_dir = os.path.join(PROJECT_ROOT, 'models')
app_dir = os.path.join(PROJECT_ROOT, 'app')

X_train = pd.read_csv(os.path.join(data_dir, 'X_train_selected.csv'))
y_train_log = pd.read_csv(os.path.join(data_dir, 'y_train_log.csv')).squeeze()

print(f'Training matrix shape : {X_train.shape}')
print(f'Target series length  : {len(y_train_log)}')

metadata = build_and_save_model_bundle(
    X_train=X_train,
    y_train_log=y_train_log,
    artifact_dir=models_dir,
    bundle_filename='ames_housing_bundle.joblib',
    n_splits=5,
    random_state=42,
)

print('\nSerialized Model Bundle Metadata:')
print(json.dumps(metadata, indent=2))

Training matrix shape : (1458, 60)
Target series length  : 1458

Serialized Model Bundle Metadata:
{
  "bundle_path": "c:\\Users\\Jonathan\\Documents\\Development\\real-estate-price-prediction\\models\\ames_housing_bundle.joblib",
  "model_type": "LGBMRegressor",
  "n_features": 60,
  "cv_rmse_log": 0.13207,
  "residual_std_log": 0.1321,
  "empirical_95_interval_log": [
    -0.30704,
    0.24766
  ]
}


## 3. Local Inference & Calibrated Confidence Intervals

Before exposing the model over HTTP, we verify `predict_with_confidence()` on three representative scenarios in scaled feature space (where `0.0` corresponds to the Ames training median). Any omitted features are automatically imputed using the training medians stored inside the model bundle.

In [3]:
bundle_path = os.path.join(models_dir, 'ames_housing_bundle.joblib')
bundle = joblib.load(bundle_path)

scenarios = {
    '1. Baseline Median Property (All Medians)': {},
    '2. Renovated Upper-Tier Home': {
        'OverallQual': 1.5,
        'GrLivArea': 1.2,
        'TotalBsmtSF': 1.0,
        'GarageCars': 1.0,
    },
    '3. Compact Entry-Level Home': {
        'OverallQual': -1.0,
        'GrLivArea': -0.8,
        'TotalBsmtSF': -0.7,
    },
}

rows = []
for label, feats in scenarios.items():
    res = predict_with_confidence(bundle, input_features=feats, confidence_level=0.95)
    rows.append({
        'Scenario': label,
        'Estimated Price ($)': f"${res['estimated_price_usd']:,.2f}",
        '95% CI Lower ($)': f"${res['confidence_range_usd']['lower_bound']:,.2f}",
        '95% CI Upper ($)': f"${res['confidence_range_usd']['upper_bound']:,.2f}",
        'Log Prediction': res['log_prediction'],
    })

pd.DataFrame(rows)

,Scenario,Estimated Price ($),95% CI Lower ($),95% CI Upper ($),Log Prediction
0,1. Baseline Median Property (All Medians),"$159,623.46","$123,211.42","$206,796.08",11.98058
1,2. Renovated Upper-Tier Home,"$213,160.38","$164,536.00","$276,154.37",12.26980
2,3. Compact Entry-Level Home,"$143,085.68","$110,446.08","$185,371.01",11.87121


## 4. Deployment Artifacts Generation (`app/`)

`write_deployment_files()` generates all files required for containerized execution and Render Blueprint cloud hosting:
- `app/api.py` — FastAPI REST service (`/health`, `/features`, `/predict`)
- `app/streamlit_app.py` — Interactive Streamlit valuation tool for real estate agents and buyers
- `app/requirements_deploy.txt` — Lightweight production dependencies
- `app/Dockerfile` — Reproducible container image (`python:3.10-slim` + `libgomp1`)
- `app/render.yaml` — Multi-service Render Blueprint configuration

In [4]:
generated = write_deployment_files(
    output_dir=app_dir,
    bundle_rel_path='models/ames_housing_bundle.joblib',
)

print('Verified Deployment Artifacts:')
for name, path in generated.items():
    rel = os.path.relpath(path, PROJECT_ROOT)
    size_kb = os.path.getsize(path) / 1024
    print(f'  - {name:<25} -> {rel} ({size_kb:.2f} KB)')

Verified Deployment Artifacts:
  - api.py                    -> app\api.py (3.45 KB)
  - streamlit_app.py          -> app\streamlit_app.py (4.48 KB)
  - requirements_deploy.txt   -> app\requirements_deploy.txt (0.16 KB)
  - Dockerfile                -> app\Dockerfile (0.68 KB)
  - render.yaml               -> app\render.yaml (0.88 KB)


## 5. FastAPI Service Smoke Tests (`TestClient`)

We test the FastAPI application in-process via `fastapi.testclient.TestClient` to verify that startup model loading, input validation, and JSON response schemas work without starting an external server.

In [5]:
os.environ['MODEL_BUNDLE_PATH'] = bundle_path

try:
    from fastapi.testclient import TestClient
    from app.api import app

    with TestClient(app) as client:
        # 1. Health check
        r_health = client.get('/health')
        assert r_health.status_code == 200
        print('GET /health   ->', r_health.status_code, r_health.json())

        # 2. Feature schema check
        r_feat = client.get('/features')
        assert r_feat.status_code == 200
        feat_payload = r_feat.json()
        print(f"GET /features -> {r_feat.status_code} (n_features={feat_payload['n_features']})")

        # 3. Valuation prediction check
        sample_request = {
            'features': {'OverallQual': 1.0, 'GrLivArea': 0.8, 'GarageCars': 1.0},
            'confidence_level': 0.95,
        }
        r_pred = client.post('/predict', json=sample_request)
        assert r_pred.status_code == 200
        print('POST /predict ->', r_pred.status_code)
        print(json.dumps(r_pred.json(), indent=2))
except ImportError as exc:
    print(f'FastAPI / httpx not installed in current kernel: {exc}')

c:\Users\Jonathan\.conda\envs\real-estate-env\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


GET /health   -> 200 {'status': 'healthy', 'model_loaded': 'True'}
GET /features -> 200 (n_features=60)
POST /predict -> 200
{
  "estimated_price_usd": 181956.09,
  "confidence_level": 0.95,
  "confidence_range_usd": {
    "lower_bound": 140449.74,
    "upper_bound": 235728.5
  },
  "log_prediction": 12.11153,
  "model_type": "LGBMRegressor"
}


## 6. Cloud Deployment & Live Verification (Render)

The two web services defined in `app/render.yaml` are deployed on Render:
- **Interactive Streamlit UI:** `https://ames-housing-valuation-ui.onrender.com`
- **FastAPI REST Service (Swagger Docs):** `https://ames-housing-valuation-api.onrender.com/docs`

Set `RUN_LIVE_API_CHECK = True` below to send a live HTTP check to the deployed Render API (note: Render free-tier instances may take ~30–50 seconds to wake up after inactivity).

In [6]:
RUN_LIVE_API_CHECK = False
LIVE_API_BASE_URL = 'https://ames-housing-valuation-api.onrender.com'

if RUN_LIVE_API_CHECK:
    import requests

    health_resp = requests.get(f'{LIVE_API_BASE_URL}/health', timeout=60)
    print('Live /health:', health_resp.status_code, health_resp.json())

    pred_resp = requests.post(
        f'{LIVE_API_BASE_URL}/predict',
        json={'features': {'OverallQual': 1.0, 'GrLivArea': 0.75}, 'confidence_level': 0.95},
        timeout=60,
    )
    print('Live /predict:', pred_resp.status_code)
    print(json.dumps(pred_resp.json(), indent=2))
else:
    print('Live Render check skipped (set RUN_LIVE_API_CHECK = True to query the live cloud endpoints).')

Live Render check skipped (set RUN_LIVE_API_CHECK = True to query the live cloud endpoints).
